In [1]:
# Instalação do PySpark no Google Colab
!pip install pyspark -q

In [2]:
# Importação das bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
# Criação da sessão Spark
spark = SparkSession.builder \
    .appName("OtimizacaoModulo30") \
    .getOrCreate()

In [8]:
# Leitura dos arquivos Parquet

df_video = spark.read.parquet("/content/videos-preparados.snappy.parquet")
df_comments = spark.read.parquet("/content/videos-comments-tratados.snappy.parquet")

# Visualizando o schema de cada DataFrame
print("Schema do df_video:")
df_video.printSchema()

print("\nSchema do df_comments:")
df_comments.printSchema()

Schema do df_video:
root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)


Schema do df_comments:
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string

In [9]:
# Visualizando algumas linhas de cada DataFrame

print("Dados do df_video:")
df_video.show(5, truncate=False)

print("\nDados do df_comments:")
df_comments.show(5, truncate=False)

Dados do df_video:
+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|Title                                                                                                |Video ID   |Published At|Keyword|Likes |Comments|Views   |Interaction|Year|Month|Keyword Index|Features PCA         |Features Normal                                                 |Features                              |
+-----------------------------------------------------------------------------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+---------------------+----------------------------------------------------------------+--------------------------------------+
|ASMR 

In [10]:
# Criação das tabelas temporárias

df_video.createOrReplaceTempView("videos")
df_comments.createOrReplaceTempView("comments")

print("Tabelas temporárias criadas com sucesso!")

Tabelas temporárias criadas com sucesso!


In [11]:
# JOIN utilizando Spark SQL

join_video_comments = spark.sql("""
    SELECT *
    FROM videos v
    INNER JOIN comments c
        ON v.`Video ID` = c.`Video ID`
""")

# Visualizando o resultado do JOIN
join_video_comments.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------------------------------------------+---------------------------------+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+
|Title                             

In [12]:
# Otimização das partições

# Reparticionando os DataFrames pela coluna utilizada no JOIN.
# Isso organiza os dados de acordo com o Video ID.
df_video_repartition = df_video.repartition("Video ID")
df_comments_repartition = df_comments.repartition("Video ID")

# Reduzindo o número de partições após o processamento.
# O coalesce evita um shuffle completo dos dados.
df_video_otimizado = df_video_repartition.coalesce(4)
df_comments_otimizado = df_comments_repartition.coalesce(4)

# Criando novas tabelas temporárias com os DataFrames particionados.
df_video_otimizado.createOrReplaceTempView("videos_otimizado")
df_comments_otimizado.createOrReplaceTempView("comments_otimizado")

print("Repartition, coalesce e tabelas temporárias concluídos!")

Repartition, coalesce e tabelas temporárias concluídos!


In [13]:
# Realizando novamente o JOIN utilizando as tabelas otimizadas

join_video_comments_otimizado = spark.sql("""
    SELECT *
    FROM videos_otimizado v
    INNER JOIN comments_otimizado c
        ON v.`Video ID` = c.`Video ID`
""")

# Visualizando o resultado
join_video_comments_otimizado.show(5, truncate=False)

+---------------------------------+-----------+------------+-------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------------------------------------------------+--------------------------------------+-----------+---------------------------------+------------+-------+------+--------+--------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+
|Title                            |Video ID   |Published At|Keyword|Likes |Comments|Views   |Interaction|Year|Month|Keyword Index|Features PCA        |Features Normal                                               |Features                              |Video ID   |Title                            |Published At|Keyword|Likes |Comments|Views   |Interaction|Year|Comment                                                                                                            

In [14]:
# Analisando o plano de execução do JOIN original

join_video_comments.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [videos], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [comments], [], false

== Analyzed Logical Plan ==
Title: string, Video ID: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: int, Month: int, Keyword Index: double, Features PCA: vector, Features Normal: vector, Features: vector, Video ID: string, Title: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: string, Comment: string, Sentiment: int, ... 1 more fields
Project [Title#28, Video ID#29, Published At#30, Keyword#31, Likes#32, Comments#33, Views#34, Interaction#35, Year#36, Month#37, Keyword Index#38, Features PCA#39, Features Normal#40, Features#41, Video ID#42, Title#43, Published At#44, Keyword#45, Likes#46, Comments#47, Views#48, Interaction#49, Year#50, Comme

In [15]:
# Analisando o plano de execução do JOIN com repartition e coalesce

join_video_comments_otimizado.explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Join Inner, ('v.Video ID = 'c.Video ID)
   :- 'SubqueryAlias v
   :  +- 'UnresolvedRelation [videos_otimizado], [], false
   +- 'SubqueryAlias c
      +- 'UnresolvedRelation [comments_otimizado], [], false

== Analyzed Logical Plan ==
Title: string, Video ID: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: int, Month: int, Keyword Index: double, Features PCA: vector, Features Normal: vector, Features: vector, Video ID: string, Title: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: string, Comment: string, Sentiment: int, ... 1 more fields
Project [Title#28, Video ID#29, Published At#30, Keyword#31, Likes#32, Comments#33, Views#34, Interaction#35, Year#36, Month#37, Keyword Index#38, Features PCA#39, Features Normal#40, Features#41, Video ID#42, Title#43, Published At#44, Keyword#45, Likes#46, Comments#47, Views#48, Interactio

In [16]:
# JOIN otimizado
#
# Selecionamos somente as colunas necessárias de cada tabela.
# Isso evita carregar e duplicar informações que já estão presentes
# no DataFrame de vídeos.

df_video.createOrReplaceTempView("videos_base")
df_comments.createOrReplaceTempView("comments_base")

join_video_comments_final = spark.sql("""
    SELECT
        v.`Video ID`,
        v.Title,
        v.`Published At`,
        v.Keyword,
        v.Likes,
        v.Comments,
        v.Views,
        v.Interaction,
        v.Year,
        v.Month,
        v.`Keyword Index`,
        v.`Features PCA`,
        v.`Features Normal`,
        v.Features,
        c.Comment,
        c.Sentiment,
        c.`Likes Comment`
    FROM videos_base v
    INNER JOIN comments_base c
        ON v.`Video ID` = c.`Video ID`
    WHERE v.`Video ID` IS NOT NULL
      AND c.`Video ID` IS NOT NULL
""")

# Visualizando o resultado do JOIN otimizado
join_video_comments_final.show(5, truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------------------------------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+
|Video ID   |Title                                                                                             |Published At|Keyword|Likes|Comments|Views |Interaction|Year|Month|Keyword Index|Features PCA  

In [17]:
# Analisando o plano de execução do JOIN otimizado

join_video_comments_final.explain(True)

== Parsed Logical Plan ==
'Project ['v.Video ID, 'v.Title, 'v.Published At, 'v.Keyword, 'v.Likes, 'v.Comments, 'v.Views, 'v.Interaction, 'v.Year, 'v.Month, 'v.Keyword Index, 'v.Features PCA, 'v.Features Normal, 'v.Features, 'c.Comment, 'c.Sentiment, 'c.Likes Comment]
+- 'Filter (isnotnull('v.Video ID) AND isnotnull('c.Video ID))
   +- 'Join Inner, ('v.Video ID = 'c.Video ID)
      :- 'SubqueryAlias v
      :  +- 'UnresolvedRelation [videos_base], [], false
      +- 'SubqueryAlias c
         +- 'UnresolvedRelation [comments_base], [], false

== Analyzed Logical Plan ==
Video ID: string, Title: string, Published At: date, Keyword: string, Likes: int, Comments: int, Views: int, Interaction: int, Year: int, Month: int, Keyword Index: double, Features PCA: vector, Features Normal: vector, Features: vector, Comment: string, Sentiment: int, Likes Comment: int
Project [Video ID#29, Title#28, Published At#30, Keyword#31, Likes#32, Comments#33, Views#34, Interaction#35, Year#36, Month#37, Keywor

In [18]:
# Salvando o JOIN otimizado em formato Parquet
#
# O resultado final contém somente as colunas necessárias,
# evitando duplicação de informações entre vídeos e comentários.
#
# O formato Parquet é utilizado por ser colunar e eficiente
# para armazenamento e leitura de grandes volumes de dados.

join_video_comments_final.write \
    .mode("overwrite") \
    .parquet("/content/join-videos-comments-otimizado")

print("JOIN otimizado salvo com sucesso!")

JOIN otimizado salvo com sucesso!


In [19]:
# Verificando o resultado salvo

df_resultado = spark.read.parquet("/content/join-videos-comments-otimizado")

print("Schema do resultado final:")
df_resultado.printSchema()

print("\nPrimeiras linhas do resultado:")
df_resultado.show(5, truncate=False)

Schema do resultado final:
root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Sentiment: integer (nullable = true)
 |-- Likes Comment: integer (nullable = true)


Primeiras linhas do resultado:
+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+-------------------------

In [20]:
# Encerrando a sessão Spark após a conclusão de todas as etapas

spark.stop()